# 0 - The interactive backend, at a glance

`digitalearth`'s **interactive** backend renders pyramids rasters and vector data as live
[HoloViz](https://holoviz.org/) maps (HoloViews + GeoViews + Bokeh, with Datashader and Panel for the
big-data and dashboard layers). Everything is driven by one chainable class, `InteractiveMap`:

```python
InteractiveMap(crs=4326, title="example").field(dem, cmap="terrain").colorbar()
```

Every map below is **live** - pan, zoom, hover and the toolbar are real. **Run this notebook** (Run All /
Shift-Enter) to see them draw.

| Notebook | Covers |
|---|---|
| `01_raster` | `field`, `quadmesh`, `contours`, `rgb`, `large_image`, frozen colour scales |
| `02_big_data` | `rasterize`, `datashade`, `trajectory`, `hexbin` |
| `03_vectors` | `points`, `lines`, `polygons`, `choropleth`, `labels`, vector fields |
| `04_mesh_density` | `trimesh`, `kde` |
| `05_temporal_animation` | `timecube`, `frames`, `play`, `save_animation` |
| `06_decoration_projection` | `tiles`, `features`, `coastlines`, `projection`, `theme` |
| `07_interaction` | `hover`, `on_select`/`on_tap`/`on_reset`/`on_resize`, `draw`, `annotate`, `cross_filter` |
| `08_linked_layouts` | `panels`, `tabs`, `swipe`, `linked_views`, `grid` |
| `09_dashboards_deployment` | `layer_control`, `attribute_table`, `serve`, `share_url`, `live` |

**Setup** - load the backends.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


## Sample data

The guide uses real data bundled at `examples/data/` (acquired with the sibling **earthlens** package):

- **`germany_pop_2020_1km.tif`** / **`..._2010_...`** - gridded population of Germany (persons/km2).
- **`global/ncep_tavg_01..12.tif`** - twelve months of global mean temperature.
- **`heidelberg_birds_2023.geojson`** (3000 points) / **`heidelberg_roads.geojson`** / **`germany_adm1.geojson`**
  (the 16 federal states) - point, line and polygon vector layers.


## A first map

Read the German population raster and draw it as an interactive field. (A `clim` caps the colour at the 99th percentile so the cities do not wash everything else out.)

In [ ]:
import numpy as np

pop = Dataset.read_file(str(DATA / "germany_pop_2020_1km.tif"))
values = pop.read_array(band=0)
cap = float(np.nanpercentile(values[values > 0], 99))
print("population raster:", pop.rows, "x", pop.columns, "| EPSG:", pop.epsg, "| p99:", round(cap))

m = InteractiveMap(crs=pop.epsg, title="Germany population 2020 (persons / km2)")
m.field(pop, cmap="inferno", clim=(0.0, cap)).colorbar()
show(m)

## One-liner: `quickmap`

`quickmap` picks the builder from the data and returns a ready map; `backend="interactive"` selects this
tier.

In [ ]:
from digitalearth import quickmap

m = quickmap(pop, crs=pop.epsg, backend="interactive")
show(m)

## Theming

`theme()` sets the Bokeh renderer theme for every layer - `"dark_minimal"`, `"caliber"`, `"night_sky"`,
`"contrast"`, or `None` to restore the default.

In [ ]:
m = InteractiveMap(crs=pop.epsg, title="dark theme")
m.theme("dark_minimal").field(pop, cmap="magma", clim=(0.0, cap)).colorbar()
show(m)

Open the following notebooks in order, or jump to the capability you need.